In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!wget -q -O "YT-BB-Dog.zip" "https://www.lirmm.fr/YT-BB-Dog_Sibetan/files/YT-BB-dog.zip"
!ls -lh YT-BB-Dog.zip

-rw-r--r-- 1 root root 583M Oct 28  2025 YT-BB-Dog.zip


In [ ]:
!unzip -q "YT-BB-Dog.zip" -d "dataset"
!find dataset -maxdepth 3 | head -20

dataset
dataset/YT-BB-dog
dataset/YT-BB-dog/YT-BB-Dog_random_bckg.zip
dataset/YT-BB-dog/YT-BB-Dog.zip


In [ ]:
!unzip -q "dataset/YT-BB-dog/YT-BB-Dog.zip" -d "dataset/perros"
!find dataset/perros -maxdepth 2 | head -10

dataset/perros
dataset/perros/YT-BB-Dog
dataset/perros/YT-BB-Dog/test
dataset/perros/YT-BB-Dog/train


In [ ]:
import os
from glob import glob

base = "dataset/perros/YT-BB-Dog"

for parte in ["train", "test"]:
    ruta = os.path.join(base, parte)
    perros = [d for d in os.listdir(ruta) if os.path.isdir(os.path.join(ruta, d))]
    imagenes = glob(os.path.join(ruta, "*", "*.jpg"))
    print(f"{parte.upper()}:")
    print(f"  Perros: {len(perros)}")
    print(f"  Imágenes: {len(imagenes)}")

TRAIN:
  Perros: 2000
  Imágenes: 19932
TEST:
  Perros: 723
  Imágenes: 7104


In [ ]:

import os
import random
from glob import glob
import pandas as pd

random.seed(42)

base = "dataset/perros/YT-BB-Dog"

def obtener_perros(ruta):
    perros = []
    for nombre in os.listdir(ruta):
        carpeta = os.path.join(ruta, nombre)
        if os.path.isdir(carpeta):
            imagenes = glob(os.path.join(carpeta, "*.jpg"))
            if len(imagenes) >= 2:
                perros.append((nombre, imagenes))
    return perros

def crear_pares(perros, pares_por_perro):
    pares = []

    # Mismo perro (1)
    for nombre, imagenes in perros:
        for _ in range(pares_por_perro):
            img1, img2 = random.sample(imagenes, 2)
            pares.append({
                "imagen1": img1,
                "imagen2": img2,
                "label": 1
            })

    # Perros diferentes (0)
    cantidad = len(pares)

    for _ in range(cantidad):
        perro1, imagenes1 = random.choice(perros)
        perro2, imagenes2 = random.choice(perros)

        while perro1 == perro2:
            perro2, imagenes2 = random.choice(perros)

        pares.append({
            "imagen1": random.choice(imagenes1),
            "imagen2": random.choice(imagenes2),
            "label": 0
        })

    df = pd.DataFrame(pares)
    return df.sample(frac=1, random_state=42).reset_index(drop=True)


# Obtener perros de entrenamiento y prueba
todos_train = obtener_perros(os.path.join(base, "train"))
todos_test = obtener_perros(os.path.join(base, "test"))

# Seleccionar 1100 perros diferentes del conjunto train
perros_seleccionados = random.sample(todos_train, 1100)

# Separar identidades para entrenamiento y validación
perros_train = perros_seleccionados[:1000]
perros_val = perros_seleccionados[1000:]

# Seleccionar 100 perros del conjunto test
perros_test = random.sample(todos_test, 100)

# Crear pares independientes
df_train = crear_pares(perros_train, 3)
df_val = crear_pares(perros_val, 10)
df_test = crear_pares(perros_test, 10)

# Guardar los pares
os.makedirs("pares", exist_ok=True)

df_train.to_csv("pares/pares_train.csv", index=False)
df_val.to_csv("pares/pares_val.csv", index=False)
df_test.to_csv("pares/pares_test.csv", index=False)

print("ENTRENAMIENTO:", len(df_train), "pares")
print("VALIDACIÓN:", len(df_val), "pares")
print("PRUEBA:", len(df_test), "pares")

print("\nPerros utilizados:")
print("Entrenamiento:", len(perros_train))
print("Validación:", len(perros_val))
print("Prueba:", len(perros_test))

print("\nComprobación de identidades:")
print("Train y validación separados:",
      set(p[0] for p in perros_train).isdisjoint(
          set(p[0] for p in perros_val)
      ))


ENTRENAMIENTO: 6000 pares
VALIDACIÓN: 2000 pares
PRUEBA: 2000 pares

Perros utilizados:
Entrenamiento: 1000
Validación: 100
Prueba: 100

Comprobación de identidades:
Train y validación separados: True


In [ ]:

import cv2
import numpy as np
import pandas as pd

TAMANO = 128

def cargar_imagen(ruta):
    imagen = cv2.imread(ruta)
    if imagen is None:
        raise ValueError(f"No se pudo cargar: {ruta}")
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    imagen = cv2.resize(imagen, (TAMANO, TAMANO))
    return imagen.astype("float32") / 255.0

def preparar_datos(df):
    X1, X2, y = [], [], []
    for _, fila in df.iterrows():
        X1.append(cargar_imagen(fila["imagen1"]))
        X2.append(cargar_imagen(fila["imagen2"]))
        y.append(fila["label"])
    return np.array(X1), np.array(X2), np.array(y, dtype="float32")

# Cargar los tres grupos de pares
df_train = pd.read_csv("pares/pares_train.csv")
df_val = pd.read_csv("pares/pares_val.csv")
df_test = pd.read_csv("pares/pares_test.csv")

# Preparar las imágenes
X1_train, X2_train, y_train = preparar_datos(df_train)
X1_val, X2_val, y_val = preparar_datos(df_val)
X1_test, X2_test, y_test = preparar_datos(df_test)

print("ENTRENAMIENTO:", X1_train.shape, X2_train.shape, y_train.shape)
print("VALIDACIÓN:   ", X1_val.shape, X2_val.shape, y_val.shape)
print("PRUEBA:       ", X1_test.shape, X2_test.shape, y_test.shape)


ENTRENAMIENTO: (6000, 128, 128, 3) (6000, 128, 128, 3) (6000,)
VALIDACIÓN:    (2000, 128, 128, 3) (2000, 128, 128, 3) (2000,)
PRUEBA:        (2000, 128, 128, 3) (2000, 128, 128, 3) (2000,)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

@tf.keras.utils.register_keras_serializable(package="Perros")
class AbsDifference(layers.Layer):
    def call(self, inputs):
        return tf.abs(inputs[0] - inputs[1])

def bloque(x, filtros):
    x = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return x

# --- Parte que "mira" una foto y saca la huella ---
entrada = layers.Input(shape=(128, 128, 3))

x = layers.RandomFlip("horizontal")(entrada)
x = layers.RandomRotation(0.05)(x)
x = layers.RandomZoom(0.1)(x)

for filtros in [32, 64, 128, 256]:
    x = bloque(x, filtros)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)

extractor = Model(entrada, x, name="extractor_128")

# --- Red siamesa completa ---
imagen_1 = layers.Input(shape=(128, 128, 3), name="imagen_1")
imagen_2 = layers.Input(shape=(128, 128, 3), name="imagen_2")

diferencia = AbsDifference(name="diferencia")(
    [extractor(imagen_1), extractor(imagen_2)]
)
x = layers.Dense(64, activation="relu", name="comparacion")(diferencia)
salida = layers.Dense(1, activation="sigmoid", name="salida")(x)

modelo = Model([imagen_1, imagen_2], salida)

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

modelo.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ imagen_1            │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ imagen_2            │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ extractor_128       │ (None, 128)       │    422,752 │ imagen_1[0][0],   │
│ (Functional)        │                   │            │ imagen_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diferencia          │ (None, 128)       │          0 │ extractor_128[0]… │
│ (AbsDifference)     │                   │            │ extractor_128[1]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ comparacion (Dense) │ (None, 64)        │      8,256 │ diferencia[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ salida (Dense)      │ (None, 1)         │         65 │ comparacion[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 431,073 (1.64 MB)

 Trainable params: 430,113 (1.64 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    ),
]

historia = modelo.fit(
    [X1_train, X2_train],
    y_train,
    validation_data=([X1_val, X2_val], y_val),
    epochs=40,
    batch_size=32,
    callbacks=callbacks,
)


Epoch 1/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 23s 99ms/step - accuracy: 0.7623 - loss: 0.5050 - val_accuracy: 0.6940 - val_loss: 0.6225 - learning_rate: 3.0000e-04
Epoch 2/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.8035 - loss: 0.4453 - val_accuracy: 0.5290 - val_loss: 0.7859 - learning_rate: 3.0000e-04
Epoch 3/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.8177 - loss: 0.4176 - val_accuracy: 0.5430 - val_loss: 0.7445 - learning_rate: 3.0000e-04
Epoch 4/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - accuracy: 0.8258 - loss: 0.4092 - val_accuracy: 0.5445 - val_loss: 0.7455 - learning_rate: 3.0000e-04
Epoch 5/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 91ms/step - accuracy: 0.8365 - loss: 0.3820 - val_accuracy: 0.5575 - val_loss: 0.6960 - learning_rate: 1.5000e-04
Epoch 6/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - accuracy: 0.8510 - loss: 0.3496 - val_accuracy: 0.5440 - val_loss: 0.7641 - learning_rate: 1.5000e-04


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# Evaluar el modelo con los perros de prueba
predicciones = modelo.predict([X1_test, X2_test]).ravel()
predicciones_clase = (predicciones >= 0.5).astype(int)

print("RESULTADOS EN PERROS NUEVOS (2000 pares)")
print("Accuracy :", round(accuracy_score(y_test, predicciones_clase), 4))
print("Precision:", round(precision_score(y_test, predicciones_clase, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, predicciones_clase, zero_division=0), 4))
print("F1-score :", round(f1_score(y_test, predicciones_clase, zero_division=0), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, predicciones_clase))


63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step
RESULTADOS EN PERROS NUEVOS (2000 pares)
Accuracy : 0.702
Precision: 0.7658
Recall   : 0.582
F1-score : 0.6614

Matriz de confusión:
[[822 178]
 [418 582]]


In [ ]:

import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# Buscar el mejor umbral utilizando VALIDACIÓN
pred_val = modelo.predict([X1_val, X2_val]).ravel()

mejor_umbral = 0.5
mejor_f1 = 0

for umbral in np.arange(0.05, 0.96, 0.05):
    clases_val = (pred_val >= umbral).astype(int)
    f1 = f1_score(y_val, clases_val, zero_division=0)

    print(f"Umbral {umbral:.2f} -> F1 {f1:.4f}")

    if f1 > mejor_f1:
        mejor_umbral = umbral
        mejor_f1 = f1

print(f"\nMejor umbral: {mejor_umbral:.2f}")
print(f"Mejor F1 en validación: {mejor_f1:.4f}")

# Evaluar una sola vez con los perros de TEST
clase = (predicciones >= mejor_umbral).astype(int)

print("\nRESULTADOS EN TEST CON EL NUEVO UMBRAL")
print("Accuracy :", round(accuracy_score(y_test, clase), 4))
print("Precision:", round(precision_score(y_test, clase, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, clase, zero_division=0), 4))
print("F1-score :", round(f1_score(y_test, clase, zero_division=0), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, clase))


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step
Umbral 0.05 -> F1 0.6667
Umbral 0.10 -> F1 0.6671
Umbral 0.15 -> F1 0.6700
Umbral 0.20 -> F1 0.6741
Umbral 0.25 -> F1 0.6848
Umbral 0.30 -> F1 0.7017
Umbral 0.35 -> F1 0.7255
Umbral 0.40 -> F1 0.7469
Umbral 0.45 -> F1 0.7683
Umbral 0.50 -> F1 0.6450
Umbral 0.55 -> F1 0.0731
Umbral 0.60 -> F1 0.0000
Umbral 0.65 -> F1 0.0000
Umbral 0.70 -> F1 0.0000
Umbral 0.75 -> F1 0.0000
Umbral 0.80 -> F1 0.0000
Umbral 0.85 -> F1 0.0000
Umbral 0.90 -> F1 0.0000
Umbral 0.95 -> F1 0.0000

Mejor umbral: 0.45
Mejor F1 en validación: 0.7683

RESULTADOS EN TEST CON EL NUEVO UMBRAL
Accuracy : 0.738
Precision: 0.696
Recall   : 0.845
F1-score : 0.7633

Matriz de confusión:
[[631 369]
 [155 845]]


In [ ]:
import os
from google.colab import files

modelo.save("modelo_perros_bolivar_scratch.keras")

ruta = "modelo_perros_bolivar_scratch.keras"
print("Existe:", os.path.exists(ruta))
print("Tamaño:", round(os.path.getsize(ruta) / (1024 * 1024), 2), "MB")

files.download(ruta)

Existe: True
Tamaño: 5.03 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

def similitud_coseno(a, b):
    num = np.sum(a * b, axis=1)
    den = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return num / np.maximum(den, 1e-9)

# Obtener las características de las imágenes de validación
sim_val = similitud_coseno(
    extractor.predict(X1_val, verbose=0),
    extractor.predict(X2_val, verbose=0),
)

# Obtener las características de las imágenes de prueba
sim_test = similitud_coseno(
    extractor.predict(X1_test, verbose=0),
    extractor.predict(X2_test, verbose=0),
)

# Buscar el mejor umbral usando validación
mejor_umbral = 0.5
mejor_f1 = 0

for umbral in np.arange(0.30, 1.00, 0.01):
    clases_val = (sim_val >= umbral).astype(int)
    f1 = f1_score(y_val, clases_val, zero_division=0)

    if f1 > mejor_f1:
        mejor_umbral = umbral
        mejor_f1 = f1

print(f"Mejor umbral de similitud: {mejor_umbral:.2f}")
print(f"F1 en validación: {mejor_f1:.4f}")

# Evaluar con perros nuevos
clase = (sim_test >= mejor_umbral).astype(int)

print("\nRESULTADOS EN TEST CON SIMILITUD COSENO")
print("Accuracy :", round(accuracy_score(y_test, clase), 4))
print("Precision:", round(precision_score(y_test, clase, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, clase, zero_division=0), 4))
print("F1-score :", round(f1_score(y_test, clase, zero_division=0), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, clase))


Mejor umbral de similitud: 0.98
F1 en validación: 0.7946

RESULTADOS EN TEST CON SIMILITUD COSENO
Accuracy : 0.821
Precision: 0.8141
Recall   : 0.832
F1-score : 0.8229

Matriz de confusión:
[[810 190]
 [168 832]]


In [ ]:
import cv2
import numpy as np

TAMANO = 128

def cargar_imagen(ruta):
    imagen = cv2.imread(ruta)
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    imagen = cv2.resize(imagen, (TAMANO, TAMANO))
    return imagen.astype("float32") / 255.0

In [ ]:
import random
import numpy as np

random.seed(7)
perros_ranking = random.sample(todos_test, 100)   # perros que la red nunca vio

galeria_imgs, consultas_imgs = [], []
for nombre, imagenes in perros_ranking:
    foto_registrada, foto_busqueda = random.sample(imagenes, 2)
    galeria_imgs.append(cargar_imagen(foto_registrada))
    consultas_imgs.append(cargar_imagen(foto_busqueda))

galeria = extractor.predict(np.array(galeria_imgs), verbose=0)
consultas = extractor.predict(np.array(consultas_imgs), verbose=0)

def normalizar(m):
    return m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)

similitudes = normalizar(consultas) @ normalizar(galeria).T   # 100 x 100

top1 = top3 = top5 = 0
for i in range(len(similitudes)):
    orden = np.argsort(-similitudes[i])
    posicion = int(np.where(orden == i)[0][0])
    top1 += posicion < 1
    top3 += posicion < 3
    top5 += posicion < 5

print("Entre 100 perros registrados, buscando con otra foto del mismo perro:")
print(f"  Queda #1:           {top1} de 100")
print(f"  Queda entre los 3:  {top3} de 100")
print(f"  Queda entre los 5:  {top5} de 100")
print("(Al azar sería: 1, 3 y 5 de 100)")

Entre 100 perros registrados, buscando con otra foto del mismo perro:
  Queda #1:           31 de 100
  Queda entre los 3:  51 de 100
  Queda entre los 5:  61 de 100
(Al azar sería: 1, 3 y 5 de 100)


In [ ]:
# No es necesario ejecutar esta celda.
# El dataset YT-BB-Dog ya fue descargado y descomprimido anteriormente.
!wget -q -O "YT-BB-Dog.zip" "https://www.lirmm.fr/YT-BB-Dog_Sibetan/files/YT-BB-dog.zip"
!unzip -q "YT-BB-Dog.zip" -d "dataset"
!unzip -q "dataset/YT-BB-dog/YT-BB-Dog.zip" -d "dataset/perros"
!ls dataset/perros/YT-BB-Dog

replace dataset/YT-BB-dog/YT-BB-Dog.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: replace dataset/perros/YT-BB-Dog/test/2000/2000_10.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: test  train


In [ ]:
import os
print(os.path.exists("dataset/perros/YT-BB-Dog"))

True


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from google.colab import files

@tf.keras.utils.register_keras_serializable(package="Perros")
class AbsDifference(layers.Layer):
    def call(self, inputs):
        return tf.abs(inputs[0] - inputs[1])

subido = files.upload()
nombre_modelo = list(subido.keys())[0]

modelo = tf.keras.models.load_model(
    nombre_modelo,
    custom_objects={"AbsDifference": AbsDifference},
    safe_mode=False,
    compile=False,
)
extractor = modelo.get_layer("extractor_128")
print("Modelo cargado:", nombre_modelo)
print("Salida del extractor:", extractor.output_shape)

Saving modelo_C_1000perros_lr0003.keras to modelo_C_1000perros_lr0003.keras
Modelo cargado: modelo_C_1000perros_lr0003.keras
Salida del extractor: (None, 128)


In [ ]:
print(modelo.get_layer("salida").get_config())

{'name': 'salida', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 1, 'activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'quantization_config': None}


In [ ]:
print("Entrada del modelo:", modelo.input_shape)
print("Salida del modelo:", modelo.output_shape)
print("Capas del modelo:", modelo.layers[-5:])

Entrada del modelo: [(None, 128, 128, 3), (None, 128, 128, 3)]
Salida del modelo: (None, 1)
Capas del modelo: [<InputLayer name=imagen_2, built=True>, <Functional name=extractor_128, built=True>, <AbsDifference name=diferencia, built=True>, <Dense name=comparacion, built=True>, <Dense name=salida, built=True>]


In [ ]:
import os
import random
from glob import glob
import cv2
import numpy as np

TAMANO = 128
base = "dataset/perros/YT-BB-Dog"

def cargar_imagen(ruta):
    imagen = cv2.imread(ruta)
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    imagen = cv2.resize(imagen, (TAMANO, TAMANO))
    return imagen.astype("float32") / 255.0

def obtener_perros(ruta):
    perros = []
    for nombre in os.listdir(ruta):
        carpeta = os.path.join(ruta, nombre)
        if os.path.isdir(carpeta):
            imagenes = glob(os.path.join(carpeta, "*.jpg"))
            if len(imagenes) >= 2:
                perros.append((nombre, imagenes))
    return perros

# Perros de la carpeta test: la red nunca los vio al entrenar
todos_test = obtener_perros(os.path.join(base, "test"))

random.seed(7)
perros_ranking = random.sample(todos_test, 100)

galeria_imgs, consultas_imgs = [], []
for nombre, imagenes in perros_ranking:
    foto_registrada, foto_busqueda = random.sample(imagenes, 2)
    galeria_imgs.append(cargar_imagen(foto_registrada))
    consultas_imgs.append(cargar_imagen(foto_busqueda))

galeria = extractor.predict(np.array(galeria_imgs), verbose=0)
consultas = extractor.predict(np.array(consultas_imgs), verbose=0)

def normalizar(m):
    return m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)

similitudes = normalizar(consultas) @ normalizar(galeria).T   # 100 x 100

top1 = top3 = top5 = 0
for i in range(len(similitudes)):
    orden = np.argsort(-similitudes[i])
    posicion = int(np.where(orden == i)[0][0])
    top1 += posicion < 1
    top3 += posicion < 3
    top5 += posicion < 5

print("Entre 100 perros registrados, buscando con otra foto del mismo perro:")
print(f"  Queda #1:           {top1} de 100")
print(f"  Queda entre los 3:  {top3} de 100")
print(f"  Queda entre los 5:  {top5} de 100")
print("(Al azar sería: 1, 3 y 5 de 100)")

Entre 100 perros registrados, buscando con otra foto del mismo perro:
  Queda #1:           70 de 100
  Queda entre los 3:  81 de 100
  Queda entre los 5:  85 de 100
(Al azar sería: 1, 3 y 5 de 100)


In [ ]:
import numpy as np
import random
import cv2
import tensorflow as tf
from tensorflow.keras import layers, Model

TAMANO = 128
P, K = 16, 4          # 16 perros por tanda, 4 fotos de cada uno

def cargar_imagen(ruta):
    imagen = cv2.imread(ruta)
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    imagen = cv2.resize(imagen, (TAMANO, TAMANO))
    return imagen.astype("float32") / 255.0

def generador():
    while True:
        elegidos = random.sample(range(len(perros_train)), P)
        imgs, etiquetas = [], []
        for idx in elegidos:
            nombre, fotos = perros_train[idx]
            if len(fotos) >= K:
                seleccion = random.sample(fotos, K)
            else:
                seleccion = random.choices(fotos, k=K)
            for f in seleccion:
                imgs.append(cargar_imagen(f))
                etiquetas.append(idx)
        yield np.array(imgs), np.array(etiquetas, dtype="int32")

dataset_tf = tf.data.Dataset.from_generator(
    generador,
    output_signature=(
        tf.TensorSpec(shape=(P * K, TAMANO, TAMANO, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(P * K,), dtype=tf.int32),
    ),
).prefetch(4)

def perdida_trillizos(etiquetas, embeddings, margen=0.3):
    etiquetas = tf.reshape(etiquetas, [-1])
    emb = tf.math.l2_normalize(embeddings, axis=1)
    dist = 1.0 - tf.matmul(emb, emb, transpose_b=True)
    mismo = tf.equal(etiquetas[:, None], etiquetas[None, :])
    yo = tf.eye(tf.shape(etiquetas)[0], dtype=tf.bool)
    mask_pos = tf.logical_and(mismo, tf.logical_not(yo))
    mask_neg = tf.logical_not(mismo)
    pos = tf.reduce_max(tf.where(mask_pos, dist, tf.zeros_like(dist)), axis=1)
    neg = tf.reduce_min(tf.where(mask_neg, dist, 10.0 * tf.ones_like(dist)), axis=1)
    return tf.reduce_mean(tf.maximum(pos - neg + margen, 0.0))

def bloque(x, filtros):
    x = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return x

entrada = layers.Input(shape=(128, 128, 3))
x = layers.RandomFlip("horizontal")(entrada)
x = layers.RandomRotation(0.05)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.RandomBrightness(0.2, value_range=(0, 1))(x)
for filtros in [32, 64, 128, 256]:
    x = bloque(x, filtros)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128)(x)
extractor = Model(entrada, x, name="extractor_128")

extractor.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=perdida_trillizos)
extractor.fit(dataset_tf, steps_per_epoch=100, epochs=30)

Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 85ms/step - loss: 0.3195
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 99ms/step - loss: 0.3047
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 93ms/step - loss: 0.3032
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - loss: 0.3023
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - loss: 0.3018
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - loss: 0.3015
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - loss: 0.3013
Epoch 8/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 87ms/step - loss: 0.3011
Epoch 9/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 89ms/step - loss: 0.3009
Epoch 10/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - loss: 0.3008
Epoch 11/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - loss: 0.3007
Epoch 12/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - loss: 0.3006
Epoch 13/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 93ms/step - loss: 0.3006
Epoch 14/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - loss: 0.3005
Epoch 15/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import os
from google.colab import files

extractor.save("extractor_D_trillizos.keras")

ruta = "extractor_D_trillizos.keras"
print("Existe:", os.path.exists(ruta))
print("Tamaño:", round(os.path.getsize(ruta) / (1024 * 1024), 2), "MB")

files.download(ruta)

Existe: True
Tamaño: 4.92 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

df_val = pd.read_csv("pares/pares_val.csv")
A = np.array([cargar_imagen(r) for r in df_val["imagen1"]])
B = np.array([cargar_imagen(r) for r in df_val["imagen2"]])
y = df_val["label"].values

ea = extractor.predict(A, verbose=0)
eb = extractor.predict(B, verbose=0)
sim = (ea * eb).sum(axis=1) / np.maximum(
    np.linalg.norm(ea, axis=1) * np.linalg.norm(eb, axis=1), 1e-9)

mejor_umbral, mejor_f1 = 0.5, 0
for u in np.arange(0.10, 1.00, 0.01):
    f1 = f1_score(y, (sim >= u).astype(int), zero_division=0)
    if f1 > mejor_f1:
        mejor_umbral, mejor_f1 = u, f1

print(f"Mejor umbral de similitud: {mejor_umbral:.2f}")
print(f"F1 en validación: {mejor_f1:.4f}")

Mejor umbral de similitud: 0.10
F1 en validación: 0.6667


In [ ]:
mejor_umbral, mejor_f1 = 0.0, 0
for u in np.arange(-1.0, 1.0, 0.01):
    f1 = f1_score(y, (sim >= u).astype(int), zero_division=0)
    if f1 > mejor_f1:
        mejor_umbral, mejor_f1 = u, f1

print(f"Mejor umbral de similitud: {mejor_umbral:.2f}")
print(f"F1 en validación: {mejor_f1:.4f}")
print("Similitud media, mismo perro:      ", round(float(sim[y == 1].mean()), 3))
print("Similitud media, perros distintos: ", round(float(sim[y == 0].mean()), 3))

Mejor umbral de similitud: -1.00
F1 en validación: 0.6667
Similitud media, mismo perro:       1.0
Similitud media, perros distintos:  1.0


In [ ]:
mu = np.concatenate([ea, eb]).mean(axis=0)

def coseno(a, b):
    return (a * b).sum(axis=1) / np.maximum(
        np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1), 1e-9)

sim_c = coseno(ea - mu, eb - mu)

def mejor(s):
    mejor_u, mejor_f = 0, 0
    for u in np.linspace(s.min(), s.max(), 1000):
        f = f1_score(y, (s >= u).astype(int), zero_division=0)
        if f > mejor_f:
            mejor_u, mejor_f = u, f
    return mejor_u, mejor_f

print("Sin centrar: min", float(sim.min()), "max", float(sim.max()))
u0, f0 = mejor(sim)
print(f"Sin centrar -> umbral {u0:.6f}, F1 {f0:.4f}")
u1, f1c = mejor(sim_c)
print(f"Centrado    -> umbral {u1:.4f}, F1 {f1c:.4f}")
print("Centrado, similitud media mismo perro:      ", round(float(sim_c[y == 1].mean()), 3))
print("Centrado, similitud media perros distintos: ", round(float(sim_c[y == 0].mean()), 3))

g = galeria - mu
c = consultas - mu
s2 = normalizar(c) @ normalizar(g).T
print("Ranking centrado, Queda #1:", sum(int(np.argmax(s2[i]) == i) for i in range(100)), "de 100")

np.save("media_D.npy", mu)

Sin centrar: min 0.9988306164741516 max 0.9999985098838806
Sin centrar -> umbral 0.999876, F1 0.9064
Centrado    -> umbral -0.8990, F1 0.7300
Centrado, similitud media mismo perro:       0.649
Centrado, similitud media perros distintos:  -0.015
Ranking centrado, Queda #1: 48 de 100


In [ ]:
df_test = pd.read_csv("pares/pares_test.csv")
At = np.array([cargar_imagen(r) for r in df_test["imagen1"]])
Bt = np.array([cargar_imagen(r) for r in df_test["imagen2"]])
yt = df_test["label"].values

sim_t = coseno(extractor.predict(At, verbose=0), extractor.predict(Bt, verbose=0))
clase = (sim_t >= u0).astype(int)

from sklearn.metrics import accuracy_score
print("Accuracy en prueba:", round(accuracy_score(yt, clase), 4))
print("F1 en prueba:      ", round(f1_score(yt, clase, zero_division=0), 4))

Accuracy en prueba: 0.8785
F1 en prueba:       0.8778


## Nota sobre las celdas de umbral de más arriba

Algunas celdas anteriores muestran "F1 0,6667" y similitud 1,0. Son intentos fallidos que se dejan a propósito como parte del ciclo de mejora. Con la pérdida de trillizos todas las similitudes quedan entre 0,9988 y 0,99999, así que buscar el umbral entre 0,10 y 0,99 (o entre -1 y 1) no sirve. La búsqueda correcta fue entre el mínimo y el máximo reales de las similitudes: umbral 0,999876, F1 de validación 0,9064.

También se probó restarle la media a los vectores (centrado): el F1 de validación bajó a 0,73 y el buscador acertó 48 de 100, así que se descartó.

Resultado final del modelo D en perros nunca vistos: accuracy 87,85 % y F1 87,78 %.